In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load Data

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
SAVE_DIR = 'figures/'
df = pd.read_excel(DATA_PATH, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head()

In [ ]:
df.rename(columns={'#':'ID',
                   'תאריך בדיקות דם': 'blood_test_date',
                   'שעת לקיחת דם':'blood_taken_time',
                   'מספר מנות':'num_of_injections',
                   'גיל הפסקת מחזור':'menopause',
                   'אסטרוגן':'estorgen',
                   'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                   'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                   'p-tau_ new_#1':'p-tau_new_#1',
                   'ביקורת אחרי שנה':'checkup after a year',
                   'חיסון':'vaccine',
                   'תרופות':'medications',
                   'הערות':'notes',
                   'Number_of_risk_faktors':'Number_of_risk_factors',
                   'P-tau217':'p-tau217',
                   'צרבונין':'Cerebonin',
                   }, inplace=True)

In [ ]:
df = df.apply(pd.to_numeric, errors='ignore')

In [ ]:
(df['p-tau217'] > 0.444).value_counts()
(df[df['Gender'] == 'f']['p-tau217'] > 0.444).value_counts()

## Females only analysis

In [ ]:
df = df[df['Gender'] == 'f']
print(df.shape)

# Settings

In [ ]:
# Graphs Settings
healthy_color = '#b4cde3' # blue
risk_color = '#fbb4ae' # red
inter_color = '#fed9a6'
pie_colors = [healthy_color, inter_color, risk_color] # blue, orange, red
vline_color = 'red'

In [ ]:
TAU_217 = 'p-tau217'
TAU_CAT = 'tau_217_category'
HIGH_TAU_217_THR = 0.53
INT_TAU_217_THR = 0.445

TMT_B_COL = 'Trail_B_(second)'
MBI_C_COL = 'Mild_behavioral_Impairment_Checklist'
IST_COL = 'IST'
ANEXITY_COL = 'Anxiety/depression'

MCI_THR = 26
MOCA_COL = 'MOCA_score'

In [ ]:
df = df[df[MOCA_COL] != '?']
df['MCI'] = df[MOCA_COL] < MCI_THR
df[TAU_CAT] = pd.cut(df[TAU_217], bins=[0, INT_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=['Low', 'Intermediate', 'High'])

df_high_vs_low = df[(df[TAU_217] > HIGH_TAU_217_THR) | (df[TAU_217] < INT_TAU_217_THR)]

Age Disterbuition

In [ ]:
sns.histplot(data=df, x='Age', kde=True, stat='count', element='step')

plt.legend()
plt.title('Histogram of Age Disterbuition')
plt.savefig(SAVE_DIR + 'Age_hist.pdf')


In [ ]:
sns.histplot(data=df, x=TAU_217, kde=True, stat='count', element='step')

plt.legend()
plt.title('Histogram of tau217 levels')
plt.savefig(SAVE_DIR + 'tau217_hist.pdf')


Cognitive screening and defining cognitive un-impaired (MoCA.26) and MCI (MoCA 19-26)

In [ ]:
sns.histplot(data=df.dropna(subset=MOCA_COL), x=MOCA_COL, kde=True, stat='count', element='step')
plt.vlines(MCI_THR, 0, plt.gca().get_ylim()[1], color=vline_color, label='MCI threshold', linestyles='dashed')

plt.legend(loc='upper left')
plt.title('Histogram of MoCA levels')
plt.savefig(SAVE_DIR + 'MOCA_hist.pdf')


# 2. Characteristics of the (pre-clinical) cohort

Fraction of participants with cognitive decline in spite of self-reported cognitive un-impairment (24% of self reportedly cognitive intact participants displayed MoCA score of MCI)

In [ ]:
healthy_n = (df['MCI'] == False).sum()
mci_n = (df['MCI'] == True).sum()
explode = (0.1, 0.1)
graph_colors = [healthy_color, risk_color]
plt.pie(df['MCI'].value_counts(), labels=['Cognitive\nHealthy', 'Cognitive\nImpaired'], autopct='%1.1f%%', colors=graph_colors, shadow=True, explode=explode)
# plt.legend(labels=[f'Cognitive Healthy N={healthy_n}', f'Cognitive Impaired N={mci_n}'], loc='upper right')
plt.title('Cognitive healthy vs Impaired distribution')
plt.savefig(SAVE_DIR + 'Cognitive_healthy_vs_Impaired_pie.pdf')

In [ ]:
explode = (0.1, 0.1, 0.1)
low_tau_n, intr_tau_n, high_tau_n = (df[TAU_CAT]=='Low').sum(),  (df[TAU_CAT]=='Intermediate').sum(), (df[TAU_CAT]=='High').sum()
graph_colors = [healthy_color, inter_color, risk_color]

plt.pie(df[TAU_CAT].value_counts(),
         labels=['Low Tau', 'Intermedaite Tau', 'High Tau'],
           autopct='%1.1f%%', colors=graph_colors,
             shadow=True, explode=explode)
# plt.legend(labels=[f'Low Tau N={low_tau_n}', f'Intermedaite-Low Tau N={intr_low_tau_n}', f'Intermedaite-High Tau N={intr_high_tau_n}', f'High Tau N={high_tau_n}'], loc='upper right')
# plt.legend(loc='upper right')
plt.savefig(SAVE_DIR + 'Tau_217_Discrete_distribution_pie.pdf')

# 3.	Comparison between p-tau217 high- versus low- participants

Performance in other cognitive (TMT-B, IST) and behavioral (MBI-C) tests in pre-clinical AD versus control groups (pre-clinical AD exhibit lower performance in TMT-B and IST tests and higher anxiety, indicating cognitive decline already at the pre-clinical stage]

In [ ]:
df_high_vs_low = df[(df[TAU_CAT] == 'High') | (df[TAU_CAT] == 'Low')]
df_high_vs_low[TAU_CAT] = df_high_vs_low[TAU_CAT].cat.remove_categories(['Intermediate'])
# df_high_vs_low = df[df[TAU_CAT].isin(['Low', 'High'])]
df_high_vs_low[TAU_CAT].value_counts()

In [ ]:
df_high_vs_low_tmt_b = df_high_vs_low.dropna(subset=[TMT_B_COL])
df_high_vs_low_tmt_b[TMT_B_COL] = df_high_vs_low_tmt_b[TMT_B_COL].astype(float)
# df_high_vs_low.groupby(TAU_CAT).mean()
grpah_colors = [healthy_color, risk_color]
sns.violinplot(data=df_high_vs_low_tmt_b, x=TAU_CAT, y=TMT_B_COL, palette=grpah_colors, inner='quartile', linewidth=1)

plt.title('Trail B vs Tau 217 category')

plt.savefig(SAVE_DIR + 'Trail_B_vs_Tau_217_category_violin.pdf')

In [ ]:
df_high_vs_low_ist = df_high_vs_low.dropna(subset=[IST_COL])
df_high_vs_low_ist[IST_COL] = df_high_vs_low_ist[IST_COL].astype(float)
grpah_colors = [healthy_color, risk_color]

sns.violinplot(data=df_high_vs_low_ist, x=TAU_CAT, y=IST_COL, palette=grpah_colors, inner='quartile', linewidth=1)

plt.title('IST vs Tau 217 category')

plt.savefig(SAVE_DIR + 'IST_vs_Tau_217_category_violin.pdf')

In [ ]:
df_high_vs_low_mbi = df_high_vs_low.dropna(subset=[MBI_C_COL])
df_high_vs_low_mbi[MBI_C_COL] = df_high_vs_low_mbi[MBI_C_COL].astype(float)
grpah_colors = [healthy_color, risk_color]

sns.violinplot(data=df_high_vs_low_mbi, x=TAU_CAT, y=MBI_C_COL, palette=grpah_colors, inner='quartile', linewidth=1)

plt.title('MBI-C vs Tau 217 category')

plt.savefig(SAVE_DIR + 'MBI_C_vs_Tau_217_category_violin.pdf')

In [ ]:
df_high_vs_low_anx = df_high_vs_low.dropna(subset=[ANEXITY_COL])
df_high_vs_low_anx[ANEXITY_COL].replace('NO', 0, inplace=True)
df_high_vs_low_anx[ANEXITY_COL].replace('YES', 1, inplace=True)
grpah_colors = [healthy_color, risk_color] * 2

yes_low_n = ((df_high_vs_low_anx[ANEXITY_COL] == 1) & (df_high_vs_low[TAU_CAT] == 'Low')).sum()
yes_high_n = ((df_high_vs_low_anx[ANEXITY_COL] == 1) & (df_high_vs_low[TAU_CAT] == 'High')).sum()
no_low_n = ((df_high_vs_low_anx[ANEXITY_COL] == 0) & (df_high_vs_low[TAU_CAT] == 'Low')).sum()
no_high_n = ((df_high_vs_low_anx[ANEXITY_COL] == 0) & (df_high_vs_low[TAU_CAT] == 'High')).sum()

print(f'Yes Low: {yes_low_n}, Yes High: {yes_high_n}, No Low: {no_low_n}, No High: {no_high_n}')

# sns.histplot(data=df_high_vs_low_anx, x=ANEXITY_COL, stat='count', element='step', stack)
sns.histplot(data=df_high_vs_low_anx, x=ANEXITY_COL, hue=TAU_CAT, multiple='stack', bins=2, palette=grpah_colors)
plt.legend(labels=['High Tau', 'Low Tau'])
# sns.violinplot(data=df_high_vs_low_anx, x=TAU_CAT, y=ANEXITY_COL, palette=grpah_colors, inner='quartile', linewidth=1)
# sns.barplot(data=df_high_vs_low_anx, x=TAU_CAT, y=ANEXITY_COL, palette=grpah_colors)

plt.title('Anxeity & Depreesion vs Tau 217 category')

plt.savefig(SAVE_DIR + 'Anxeity_and_Depreesion_vs_Tau_217_category_violin.pdf')

Total number of risk factors (not increased in pre-clinical AD as compared to low p-tau matched controls).

In [ ]:
df_high_vs_low[TAU_CAT].value_counts()

In [ ]:
N_RISK_COL = 'Number_of_risk_factors'
df_high_vs_low_risk = df_high_vs_low.dropna(subset=[N_RISK_COL])
df_high_vs_low_risk[N_RISK_COL] = df_high_vs_low_risk[N_RISK_COL].astype(int)
grpah_colors = [healthy_color, risk_color]

# stacked histogram of number of risk factors
sns.histplot(data=df_high_vs_low_risk, x=N_RISK_COL, hue=TAU_CAT, multiple='stack', bins=9, palette=grpah_colors)
plt.legend(labels=['High Tau', 'Low Tau'])

plt.title('Number of risk factors vs Tau 217 category')
plt.savefig(SAVE_DIR + 'Number_of_risk_factors_distribution.pdf')


In [ ]:
# same  but normalized
sns.histplot(data=df_high_vs_low_risk, x=N_RISK_COL, hue=TAU_CAT, multiple='fill', bins=9, palette=grpah_colors)
plt.legend(labels=['High Tau', 'Low Tau'])
# write the number of people in each category
bin_centers = np.arange(9) + 1
for j in range(9):
    low_count = ((df_high_vs_low_risk[TAU_CAT] == 'Low') & (df_high_vs_low_risk[N_RISK_COL] == j)).sum()
    high_count = ((df_high_vs_low_risk[TAU_CAT] == 'High') & (df_high_vs_low_risk[N_RISK_COL] == j)).sum()
    total = low_count + high_count
    
    if total > 0:
        high_height = high_count / total
        plt.text(j+0.5, high_height/2, str(high_count), ha='center', va='center')
        plt.text(j+0.5, high_height + (1-high_height)/2, str(low_count), ha='center', va='center')

plt.title('Number of risk factors distribution')
plt.savefig(SAVE_DIR + 'Number_of_risk_factors_distribution_normalized.pdf')
plt.show()
